In [ ]:
#r "nuget: System.Data.SqlClient, 4.8.5"
#r "nuget: Dapper, 2.1.35"

using System;
using System.Data.SqlClient;
using Dapper;
using System.Threading.Tasks;

var connString = new SqlConnectionStringBuilder {
    DataSource = "192.168.30.118,1433",
    InitialCatalog = "MCRMamSystem",
    UserID = "sa",
    Password = "sa",
    Encrypt = true,
    TrustServerCertificate = true,
    ConnectTimeout = 5
}.ConnectionString;

try
{
    using var conn = new SqlConnection(connString);
    await conn.OpenAsync();
    Console.WriteLine("✅ SQL 連線成功");

    // === 範例：撈 FileData_History 前 10 筆 ===
    var sql = @"
        const string sql = @"
SELECT 
  h.id              AS HistoryId,
  h.file_id         AS FileId,
  h.from_storage_id AS FromStorageId,
  h.to_storage_id   AS ToStorageId,
  f.filename        AS FileName,
  f.UserBit         AS UserBit,
  s_from.location   AS FromPath,
  s_to.location     AS ToPath
FROM dbo.FileData_History h
JOIN dbo.FileData f     ON f.id       = h.file_id
JOIN dbo.Storage s_from ON s_from.id  = h.from_storage_id
JOIN dbo.Storage s_to   ON s_to.id    = h.to_storage_id
WHERE h.id IN @ids;",
    var result = await conn.QueryAsync(sql);

    foreach (var row in result)
        Console.WriteLine($"{row.HistoryId}: {row.FileName} | {row.FromPath} → {row.ToPath} ({row.status})");
}
catch (Exception ex)
{
    Console.WriteLine("❌ 連線失敗");
    Console.WriteLine(ex.Message);
}


Installed Packages Dapper, 2.1.35 System.Data.SqlClient, 4.8.5


(24,30): error CS1002: 必須是 ;

(25,7): error CS1001: 必須是識別項

(25,7): error CS1002: 必須是 ;

(26,24): error CS1002: 必須是 ;

(26,33): error CS1002: 必須是 ;

(26,33): error CS1513: 必須是 }

(27,24): error CS1002: 必須是 ;

(27,30): error CS1002: 必須是 ;

(27,30): error CS1513: 必須是 }

(28,24): error CS1002: 必須是 ;

(28,37): error CS1002: 必須是 ;

(28,37): error CS1513: 必須是 }

(29,24): error CS1002: 必須是 ;

(29,35): error CS1002: 必須是 ;

(29,35): error CS1513: 必須是 }

(30,24): error CS1002: 必須是 ;

(30,32): error CS1002: 必須是 ;

(30,32): error CS1513: 必須是 }

(31,24): error CS1002: 必須是 ;

(31,31): error CS1002: 必須是 ;

(31,31): error CS1513: 必須是 }

(32,24): error CS1002: 必須是 ;

(32,32): error CS1002: 必須是 ;

(32,32): error CS1513: 必須是 }

(33,24): error CS1002: 必須是 ;

(34,6): error CS1002: 必須是 ;

(34,28): error CS1002: 必須是 ;

(35,9): error CS1003: 語法錯誤，必須是 ','

(35,10): error CS1002: 必須是 ;

(35,25): error CS1002: 必須是 ;

(35,29): error CS1003: 語法錯誤，必須是 ','

(35,30): error CS1002: 必須是 ;

(35,50): error CS1002: 必須是 ;

Error: compilation error

: 

In [11]:
#!csharp
var connString = new SqlConnectionStringBuilder {
    DataSource = "192.168.30.118,1433",
    InitialCatalog = "MCRMamSystem",
    UserID = "sa",
    Password = "sa",
    Encrypt = true,
    TrustServerCertificate = true,
    ConnectTimeout = 5
}.ConnectionString;

try
{
    using var conn = new SqlConnection(connString);
    await conn.OpenAsync();
    Console.WriteLine("✅ SQL 連線成功");
}
catch (Exception ex)
{
    Console.WriteLine("❌ 連線失敗");
    Console.WriteLine(ex.Message);
}


Security Warning: The negotiated TLS 1.0 is an insecure protocol and is supported for backward compatibility only. The recommended protocol version is TLS 1.2 and later.
✅ SQL 連線成功



warning CS1701: 假設 'Microsoft.Data.SqlClient' 所使用的組件參考 'System.Data.Common, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' 符合 'System.Data.Common' 的識別 'System.Data.Common, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'，您可能會需要提供執行階段原則

warning CS1701: 假設 'Microsoft.Data.SqlClient' 所使用的組件參考 'System.Data.Common, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' 符合 'System.Data.Common' 的識別 'System.Data.Common, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'，您可能會需要提供執行階段原則



In [3]:
using System;
using System.IO;
using System.Threading.Tasks;
using System.Diagnostics;

// --- 直接寫邏輯 (Top-level statements) ---
string testFilePath = @"B:\2101ECBB.mxf"; 

if (!File.Exists(testFilePath))
{
    Console.WriteLine("錯誤：找不到測試檔案。");
    return;
}

Console.WriteLine($"--- 刪除安全檢測開始 ---");
bool isFree = await WaitFileFreeAsync(testFilePath, 5000);

if (isFree)
{
    try
    {
        File.Delete(testFilePath);
        Console.WriteLine("結果：成功！檔案已刪除。");
    }
    catch (Exception ex)
    {
        Console.WriteLine($"結果：失敗。刪除噴錯: {ex.Message}");
    }
}
else
{
    Console.WriteLine("結果：拒絕刪除！檔案正被佔用。");
}

// --- 將 Method 放在檔案最下方 ---
static async Task<bool> WaitFileFreeAsync(string path, int ms = 2000)
{
    var sw = Stopwatch.StartNew();
    while (sw.ElapsedMilliseconds < ms)
    {
        try
        {
            using var fs = new FileStream(path, FileMode.Open, FileAccess.ReadWrite, FileShare.None);
            return true; 
        }
        catch (IOException)
        {
            Console.WriteLine($"[{sw.ElapsedMilliseconds}ms] 偵測到佔用，重試中...");
            await Task.Delay(200);
        }
    }
    return false;
}

--- 刪除安全檢測開始 ---
s] 偵測到佔用，重試中...
s] 偵測到佔用，重試中...
s] 偵測到佔用，重試中...
s] 偵測到佔用，重試中...
s] 偵測到佔用，重試中...
[1070ms] 偵測到佔用，重試中...
[1273ms] 偵測到佔用，重試中...
[1486ms] 偵測到佔用，重試中...
[1687ms] 偵測到佔用，重試中...
[1889ms] 偵測到佔用，重試中...
[2092ms] 偵測到佔用，重試中...
[2295ms] 偵測到佔用，重試中...
[2496ms] 偵測到佔用，重試中...
[2698ms] 偵測到佔用，重試中...
[2901ms] 偵測到佔用，重試中...
[3111ms] 偵測到佔用，重試中...
[3323ms] 偵測到佔用，重試中...
[3526ms] 偵測到佔用，重試中...
[3727ms] 偵測到佔用，重試中...
[3929ms] 偵測到佔用，重試中...
[4131ms] 偵測到佔用，重試中...
[4333ms] 偵測到佔用，重試中...
[4534ms] 偵測到佔用，重試中...
[4735ms] 偵測到佔用，重試中...
[4937ms] 偵測到佔用，重試中...
結果：拒絕刪除！檔案正被佔用。


<null>